# M01 - Environment check and reproducibility

**Module:** M01 - Python and AI development environment  
**Hardware label:** CPU-only. GPU cells run only when CUDA is available.  
**Estimated time:** 2-3 hours, including reading the explanations.  
**Dataset:** the small digit dataset bundled with scikit-learn. No download is required.

This notebook is the guided practical for M01. You will inventory the machine,
detect an optional GPU, demonstrate seeded randomness, train and reload a small
classifier, time a matrix operation, verify a file checksum, and write a
machine-readable environment report.

Run every cell from top to bottom in a **fresh kernel**. If a cell fails, fix
the cause instead of skipping it. The report at the end is your evidence that
the environment works.

> This notebook does not install packages. Install the dependencies first with
> `pip install -r requirements/notebooks.txt` as described in the
> [notebook guide](../../docs/lessons/m01-environment/notebook.md).

In [1]:
from __future__ import annotations

import csv
import ctypes
import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import joblib
import matplotlib

matplotlib.use("Agg")  # render plots without requiring an interactive display
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Notebook working directory: {Path.cwd()}")
print(f"Artifacts will be written to: {(Path.cwd() / OUTPUT_DIR).resolve()}")

Notebook working directory: E:\AITopics\notebooks\foundations
Artifacts will be written to: E:\AITopics\notebooks\foundations\outputs


## 1. Runtime and package inventory

An environment is not "working" just because Python starts. Record the
interpreter, operating system, and library versions. Two learners with
"Python 3.12" can still get different results if their NumPy or scikit-learn
versions differ.

`importlib.metadata` reads the installed distribution version. It is safer than
hard-coding a version string in a report.

In [2]:
def package_version(distribution: str) -> str:
    """Return the installed version, or a clear marker when it is absent."""
    try:
        return metadata.version(distribution)
    except metadata.PackageNotFoundError:
        return "not installed"


PACKAGES = ["numpy", "scikit-learn", "matplotlib", "joblib", "ipykernel", "jupyterlab"]
package_versions = {name: package_version(name) for name in PACKAGES}

runtime = {
    "python": sys.version.split()[0],
    "implementation": platform.python_implementation(),
    "executable": sys.executable,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor() or "not reported by the OS",
    "system": platform.system(),
    "release": platform.release(),
}

print(f"{'Runtime field':<16} {'Value'}")
print("-" * 72)
for key, value in runtime.items():
    print(f"{key:<16} {value}")

print("\nInstalled packages")
print("-" * 72)
for name, version in package_versions.items():
    print(f"{name:<16} {version}")

Runtime field    Value
------------------------------------------------------------------------
python           3.12.10
implementation   CPython
executable       E:\AITopics\.venv\Scripts\python.exe
platform         Windows-11-10.0.26200-SP0
machine          AMD64
processor        Intel64 Family 6 Model 167 Stepping 1, GenuineIntel
system           Windows
release          11

Installed packages
------------------------------------------------------------------------
numpy            2.5.4
scikit-learn     1.9.1
matplotlib       3.11.2
joblib           1.6.0
ipykernel        6.31.0
jupyterlab       4.6.5


## 2. Hardware inventory

RAM and free disk space decide which labs you can run locally. The code below
uses only the standard library: `ctypes` on Windows, `sysctl` on macOS, and
`os.sysconf` on Linux. If a platform does not expose a value, the report says
`null` instead of inventing one.

In [3]:
def total_ram_gb() -> float | None:
    """Return physical RAM in GiB using standard-library calls only."""
    if sys.platform == "win32":

        class MemoryStatusEx(ctypes.Structure):
            _fields_ = [
                ("dwLength", ctypes.c_ulong),
                ("dwMemoryLoad", ctypes.c_ulong),
                ("ullTotalPhys", ctypes.c_ulonglong),
                ("ullAvailPhys", ctypes.c_ulonglong),
                ("ullTotalPageFile", ctypes.c_ulonglong),
                ("ullAvailPageFile", ctypes.c_ulonglong),
                ("ullTotalVirtual", ctypes.c_ulonglong),
                ("ullAvailVirtual", ctypes.c_ulonglong),
                ("ullAvailExtendedVirtual", ctypes.c_ulonglong),
            ]

        status = MemoryStatusEx()
        status.dwLength = ctypes.sizeof(MemoryStatusEx)
        if ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(status)):
            return round(status.ullTotalPhys / 1024**3, 2)
        return None

    if sys.platform == "darwin":
        try:
            output = subprocess.check_output(
                ["sysctl", "-n", "hw.memsize"], text=True, timeout=10
            ).strip()
            return round(int(output) / 1024**3, 2)
        except (OSError, subprocess.SubprocessError, ValueError):
            return None

    try:
        pages = os.sysconf("SC_PHYS_PAGES")
        page_size = os.sysconf("SC_PAGE_SIZE")
        return round(pages * page_size / 1024**3, 2)
    except (ValueError, OSError, AttributeError):
        return None


disk = shutil.disk_usage(Path.cwd())
hardware = {
    "cpu_logical_cores": os.cpu_count(),
    "ram_gib": total_ram_gb(),
    "disk_total_gib": round(disk.total / 1024**3, 2),
    "disk_free_gib": round(disk.free / 1024**3, 2),
}

for key, value in hardware.items():
    print(f"{key:<20} {value}")

cpu_logical_cores    16
ram_gib              79.89
disk_total_gib       112.3
disk_free_gib        82.81


## 3. Optional GPU detection

A GPU is **not required** for M01. This section answers three separate
questions:

1. Is an NVIDIA driver visible through `nvidia-smi`?
2. Is PyTorch installed?
3. Does the installed PyTorch build see a CUDA device?

The driver, the CUDA runtime bundled in the PyTorch wheel, and PyTorch itself
are different layers. A machine can have a GPU and still report
`cuda_available: false` because the CPU-only PyTorch wheel is installed. That is
a configuration fact, not a crash.

In [4]:
def nvidia_smi_info() -> dict | None:
    """Query the NVIDIA driver without importing PyTorch."""
    executable = shutil.which("nvidia-smi")
    if executable is None:
        return None

    command = [
        executable,
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ]
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=15)
    except (OSError, subprocess.SubprocessError) as exc:
        return {"error": f"{type(exc).__name__}: {exc}"}

    if result.returncode != 0:
        detail = result.stderr.strip() or f"exit code {result.returncode}"
        return {"error": detail}

    devices = []
    for line in result.stdout.strip().splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) >= 3:
            devices.append(
                {"name": parts[0], "memory_total": parts[1], "driver_version": parts[2]}
            )
    return {"devices": devices}


def detect_gpu() -> dict:
    info: dict = {
        "torch_installed": False,
        "cuda_available": False,
        "nvidia_smi": nvidia_smi_info(),
    }

    try:
        import torch
    except Exception as exc:  # a broken DLL also raises here on Windows
        info["torch_note"] = (
            f"PyTorch could not be imported ({type(exc).__name__}: {exc}). "
            "The CPU-only path does not need it."
        )
        return info

    info["torch_installed"] = True
    info["torch_version"] = torch.__version__
    info["cuda_build"] = torch.version.cuda
    info["cuda_available"] = bool(torch.cuda.is_available())

    if info["cuda_available"]:
        devices = []
        for index in range(torch.cuda.device_count()):
            properties = torch.cuda.get_device_properties(index)
            devices.append(
                {
                    "index": index,
                    "name": properties.name,
                    "vram_gib": round(properties.total_memory / 1024**3, 2),
                    "compute_capability": f"{properties.major}.{properties.minor}",
                }
            )
        info["devices"] = devices
    elif info["torch_installed"]:
        info["torch_note"] = (
            "PyTorch is installed but this build does not see CUDA. "
            "You may have the CPU-only wheel, a driver problem, or no NVIDIA GPU."
        )

    return info


gpu = detect_gpu()
print(json.dumps(gpu, indent=2))

{
  "torch_installed": false,
  "cuda_available": false,
  "nvidia_smi": {
    "devices": [
      {
        "name": "NVIDIA GeForce RTX 4060 Ti",
        "memory_total": "16380 MiB",
        "driver_version": "610.88"
      }
    ]
  },
  "torch_note": "PyTorch could not be imported (ModuleNotFoundError: No module named 'torch'). The CPU-only path does not need it."
}


## 4. Seeded randomness

Seeds make a sequence of pseudo-random numbers repeatable **on the same code,
library version, and hardware**. They do not guarantee bit-identical results
across different GPUs or BLAS builds. The demonstration below draws bootstrap
sample means twice from the same seed and compares the arrays.

`np.random.seed` controls the legacy global generator. `np.random.default_rng`
creates an independent generator, which is the recommended style for new code.
Both are set here because older libraries may still use the global generator.

In [5]:
random.seed(SEED)
np.random.seed(SEED)


def bootstrap_means(seed: int, n_resamples: int = 300, sample_size: int = 120) -> np.ndarray:
    """Return bootstrap means produced by a local, seeded generator."""
    rng = np.random.default_rng(seed)
    sample = rng.normal(loc=0.0, scale=1.0, size=sample_size)
    indices = rng.integers(0, sample_size, size=(n_resamples, sample_size))
    return sample[indices].mean(axis=1)


bootstrap_a = bootstrap_means(SEED)
bootstrap_b = bootstrap_means(SEED)
bootstrap_identical = bool(np.array_equal(bootstrap_a, bootstrap_b))

print(f"Two runs with seed {SEED} are identical: {bootstrap_identical}")
print(f"First five means: {np.round(bootstrap_a[:5], 4)}")
print(f"Mean of bootstrap means: {bootstrap_a.mean():.4f}")

assert bootstrap_identical, "The seeded generator did not reproduce its own sequence."

Two runs with seed 42 are identical: True
First five means: [-0.0569 -0.1336 -0.0662 -0.1101 -0.066 ]
Mean of bootstrap means: -0.0627


## 5. Train a small classifier

The scikit-learn **digits** dataset contains 1,797 images of handwritten digits.
Each image is 8 x 8 pixels, stored as 64 numeric features. It is bundled with
scikit-learn, so this section works offline.

The pipeline standardizes the features and fits multinomial logistic regression.
This is deliberately small: the point is to verify that numeric libraries,
plotting, and model serialization work, not to achieve a benchmark score.

In [6]:
digits = load_digits()
X, y = digits.data, digits.target

print(f"Feature matrix: {X.shape}")
print(f"Target vector: {y.shape}")
print(f"Classes: {np.unique(y).tolist()}")
print(f"Pixel range: {X.min():.1f} to {X.max():.1f}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

model = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, random_state=SEED)),
    ]
)

fit_start = time.perf_counter()
model.fit(X_train, y_train)
fit_seconds = time.perf_counter() - fit_start

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Training rows: {len(X_train)}, test rows: {len(X_test)}")
print(f"Fit time: {fit_seconds:.3f} s")
print(f"Test accuracy: {accuracy:.4f}")
print("\nClassification report")
print(classification_report(y_test, predictions, digits=3))

Feature matrix: (1797, 64)
Target vector: (1797,)
Classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Pixel range: 0.0 to 16.0
Training rows: 1347, test rows: 450
Fit time: 0.018 s
Test accuracy: 0.9778

Classification report
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        45
           1      0.935     0.935     0.935        46
           2      1.000     1.000     1.000        44
           3      1.000     1.000     1.000        46
           4      0.978     1.000     0.989        45
           5      1.000     0.978     0.989        46
           6      1.000     0.978     0.989        45
           7      0.978     1.000     0.989        45
           8      0.909     0.930     0.920        43
           9      0.977     0.956     0.966        45

    accuracy                          0.978       450
   macro avg      0.978     0.978     0.978       450
weighted avg      0.978     0.978     0.978       450



In [7]:
figure, axis = plt.subplots(figsize=(6.4, 5.6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=[str(label) for label in range(10)],
    cmap="Blues",
    colorbar=False,
    ax=axis,
)
axis.set_title(f"Digits test set (accuracy {accuracy:.3f})")
figure.tight_layout()

confusion_path = OUTPUT_DIR / "m01_confusion_matrix.png"
figure.savefig(confusion_path, dpi=150)
plt.close(figure)

print(f"Saved confusion matrix to {confusion_path.resolve()}")

Saved confusion matrix to E:\AITopics\notebooks\foundations\outputs\m01_confusion_matrix.png


## 6. Save and reload the model

A model that exists only in a notebook kernel is not a deliverable. Serialize it,
reload it in the same process, and compare predictions. `joblib` is the format
scikit-learn recommends for objects containing NumPy arrays.

Loading a pickle or joblib file executes Python code. Never load a model file
from an untrusted source.

In [8]:
model_path = OUTPUT_DIR / "m01_digits_logreg.joblib"
joblib.dump(model, model_path)

reloaded_model = joblib.load(model_path)
reloaded_predictions = reloaded_model.predict(X_test)
predictions_identical = bool(np.array_equal(predictions, reloaded_predictions))

print(f"Saved model: {model_path.resolve()}")
print(f"Model file size: {model_path.stat().st_size / 1024:.1f} KiB")
print(f"Reloaded predictions match: {predictions_identical}")

assert predictions_identical, "Reloaded model produced different predictions."

Saved model: E:\AITopics\notebooks\foundations\outputs\m01_digits_logreg.joblib
Model file size: 7.9 KiB
Reloaded predictions match: True


## 7. CPU and optional GPU timing

The operation is a 768 x 768 single-precision matrix multiplication. CPU and GPU
timings are not a fair benchmark of the whole AI workload: data transfer, small
batch sizes, and kernel launch overhead often dominate. The purpose is to show
how to measure and label a timing result honestly.

If CUDA is unavailable, the GPU branch is skipped and the report records why.

In [9]:
rng = np.random.default_rng(SEED)
matrix_size = 768
left = rng.standard_normal((matrix_size, matrix_size), dtype=np.float32)
right = rng.standard_normal((matrix_size, matrix_size), dtype=np.float32)

for _ in range(2):  # warm-up
    left @ right

cpu_repeats = 5
cpu_start = time.perf_counter()
for _ in range(cpu_repeats):
    result_cpu = left @ right
cpu_seconds = (time.perf_counter() - cpu_start) / cpu_repeats
cpu_gflops = 2 * matrix_size**3 / cpu_seconds / 1e9

timing = {
    "matrix_size": matrix_size,
    "dtype": "float32",
    "cpu_seconds": round(cpu_seconds, 6),
    "cpu_gflops": round(cpu_gflops, 2),
    "gpu_seconds": None,
    "gpu_gflops": None,
}

print(f"CPU mean time: {cpu_seconds * 1000:.2f} ms ({cpu_gflops:.2f} GFLOP/s)")

if gpu.get("cuda_available"):
    import torch

    device = torch.device("cuda")
    left_gpu = torch.tensor(left, device=device)
    right_gpu = torch.tensor(right, device=device)
    for _ in range(3):
        left_gpu @ right_gpu
    torch.cuda.synchronize()

    gpu_start = time.perf_counter()
    for _ in range(10):
        result_gpu = left_gpu @ right_gpu
    torch.cuda.synchronize()
    gpu_seconds = (time.perf_counter() - gpu_start) / 10
    gpu_gflops = 2 * matrix_size**3 / gpu_seconds / 1e9
    timing["gpu_seconds"] = round(gpu_seconds, 6)
    timing["gpu_gflops"] = round(gpu_gflops, 2)
    print(f"GPU mean time: {gpu_seconds * 1000:.2f} ms ({gpu_gflops:.2f} GFLOP/s)")
else:
    print("GPU timing skipped: CUDA is not available in this kernel.")

CPU mean time: 1.40 ms (647.84 GFLOP/s)
GPU timing skipped: CUDA is not available in this kernel.


## 8. Dataset checksums

A checksum is a fingerprint of a file. Dataset publishers usually provide a
SHA-256 value next to each download. You compute the hash of your copy and
compare it **before** opening or training on the data. A mismatch means the file
is corrupted, incomplete, or not the version you expected.

This section exports the bundled digits data to CSV, writes a SHA-256 manifest,
and verifies it. The same `verify_sha256` function works for downloaded files.
The exercise stays offline so it runs in any classroom.

In [10]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Hash a file in chunks so large datasets do not fill memory."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_sha256(path: Path, manifest_path: Path) -> tuple[bool, str, str]:
    """Compare a file against a manifest formatted as '<hash>  <filename>'."""
    expected = manifest_path.read_text(encoding="utf-8").split()[0]
    actual = sha256_file(path)
    return actual == expected, expected, actual


csv_path = OUTPUT_DIR / "digits.csv"
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow([f"pixel_{index}" for index in range(64)] + ["target"])
    for row, label in zip(X.astype(int), y, strict=True):
        writer.writerow([*row.tolist(), int(label)])

digest = sha256_file(csv_path)
manifest_path = csv_path.with_suffix(".csv.sha256")
manifest_path.write_text(f"{digest}  {csv_path.name}\n", encoding="utf-8")

checksum_ok, expected_hash, actual_hash = verify_sha256(csv_path, manifest_path)
print(f"CSV rows: {len(X)} plus header")
print(f"SHA-256: {actual_hash}")
print(f"Checksum verified: {checksum_ok}")

assert checksum_ok, "The checksum manifest did not match the exported file."

CSV rows: 1797 plus header
SHA-256: eca5c9d130b59c78d8cfd620d9eddfcf8071b75182d306e96476c350d6f0474d
Checksum verified: True


## 9. Write the environment report

The report is a JSON file that another person or script can inspect without
reading notebook output. It includes the runtime, packages, hardware, GPU status,
reproducibility result, model metrics, timing, and artifact checksums.

Commit the **report**, not the model weights or the exported CSV. Large artifacts
belong in an ignored `outputs/` directory or in release storage.

In [11]:
report = {
    "schema_version": 1,
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "course": {
        "module": "M01",
        "notebook": "m01_environment_check.ipynb",
        "seed": SEED,
        "hardware_label": "CPU-only",
    },
    "runtime": runtime,
    "packages": package_versions,
    "hardware": hardware,
    "gpu": gpu,
    "determinism": {"bootstrap_identical": bootstrap_identical},
    "model": {
        "dataset": "sklearn.datasets.load_digits",
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "test_accuracy": round(float(accuracy), 6),
        "fit_seconds": round(float(fit_seconds), 6),
        "reload_predictions_identical": predictions_identical,
    },
    "timing": timing,
    "artifacts": {
        "confusion_matrix": str(confusion_path),
        "model": str(model_path),
        "csv": str(csv_path),
        "csv_sha256": actual_hash,
        "checksum_verified": checksum_ok,
    },
}

report_path = OUTPUT_DIR / "environment_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print(f"Environment report written to {report_path.resolve()}")
print(json.dumps(report, indent=2))

Environment report written to E:\AITopics\notebooks\foundations\outputs\environment_report.json
{
  "schema_version": 1,
  "generated_utc": "2026-10-30T04:37:56+00:00",
  "course": {
    "module": "M01",
    "notebook": "m01_environment_check.ipynb",
    "seed": 42,
    "hardware_label": "CPU-only"
  },
  "runtime": {
    "python": "3.12.10",
    "implementation": "CPython",
    "executable": "E:\\AITopics\\.venv\\Scripts\\python.exe",
    "platform": "Windows-11-10.0.26200-SP0",
    "machine": "AMD64",
    "processor": "Intel64 Family 6 Model 167 Stepping 1, GenuineIntel",
    "system": "Windows",
    "release": "11"
  },
  "packages": {
    "numpy": "2.5.4",
    "scikit-learn": "1.9.1",
    "matplotlib": "3.11.2",
    "joblib": "1.6.0",
    "ipykernel": "6.31.0",
    "jupyterlab": "4.6.5"
  },
  "hardware": {
    "cpu_logical_cores": 16,
    "ram_gib": 79.89,
    "disk_total_gib": 112.3,
    "disk_free_gib": 82.81
  },
  "gpu": {
    "torch_installed": false,
    "cuda_available": fa

## 10. Check your work

Before you move to the lab, confirm all of the following:

- [ ] Every cell ran from top to bottom in a fresh kernel without an unhandled error.
- [ ] The runtime and package table lists the versions you actually used.
- [ ] The seeded bootstrap arrays were identical.
- [ ] The classifier trained and reported test accuracy.
- [ ] The saved model reloaded and produced identical predictions.
- [ ] The CPU timing completed; the GPU branch was skipped or recorded.
- [ ] The CSV checksum verified.
- [ ] `outputs/environment_report.json` exists and names your machine.

**Next:** complete the [M01 lab brief](../../docs/lessons/m01-environment/lab.md),
then use the [assessment page](../../docs/lessons/m01-environment/assessment.md)
to check your evidence.

### Reflection prompts

1. Which version or hardware detail would be hardest for you to recreate on a
   different machine, and why?
2. If `cuda_available` is false, is that a fault, a missing driver, or a CPU-only
   PyTorch wheel? What evidence separates those cases?
3. What did you learn from the timing result that a single "it works" claim would
   hide?
4. What will you record in every future lab report so another student can rerun
   your result?